In [1]:
import os
os.chdir("../")
%pwd

'C:\\Users\\Sayantan Nandi\\Desktop\\mlops\\developer-burnout-project'

In [2]:
import pandas as pd

from src.config.configuration import ConfigurationManager

In [3]:
configurationManager = ConfigurationManager()
data_ingestion_config = configurationManager.get_data_ingestion_config()
data_transformation_config = configurationManager.get_data_transformation_config()

[2026-04-16 02:41:11,847: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-16 02:41:11,848: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-16 02:41:11,849: INFO: common: created directory at: artifacts]
[2026-04-16 02:41:11,851: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-16 02:41:11,852: INFO: common: created directory at: artifacts/data_transformation]


In [4]:
data = pd.read_csv(f"{data_ingestion_config.unzip_dir}/dataset.csv")

In [5]:
data.head()

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
0,26.0,12.0,10.33,4.45,2.0,11.0,4.0,1.0,15.07,0.14,55.96,Medium
1,39.0,10.0,8.62,5.77,5.0,15.0,11.0,5.0,13.25,0.54,82.22,High
2,34.0,13.0,NaN,4.03,5.0,2.0,18.0,9.0,11.18,1.54,61.77,Medium
3,30.0,1.0,6.85,6.47,2.0,15.0,26.0,1.0,11.14,0.96,54.98,Medium
4,27.0,7.0,4.24,5.80,NaN,9.0,17.0,7.0,8.05,0.36,27.90,Low


In [6]:
data.isna().sum()

age                 140
experience_years    140
daily_work_hours    140
sleep_hours         140
caffeine_intake     140
bugs_per_day        140
commits_per_day     140
meetings_per_day    140
screen_time         140
exercise_hours      140
stress_level        140
burnout_level       140
dtype: int64

In [7]:
cols_with_nan = data.apply(lambda row: row.index[row.isna()].tolist(), axis=1)
cols_with_nan

0                       []
1                       []
2       [daily_work_hours]
3                       []
4        [caffeine_intake]
               ...        
6995                    []
6996                    []
6997                    []
6998                    []
6999                    []
Length: 7000, dtype: object

In [8]:
data["nan_cols"] = data.apply(lambda row: row.index[row.isna()].tolist(), axis=1)

In [9]:
data

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level,nan_cols
0,26.0,12.0,10.33,4.45,2.0,11.0,4.0,1.0,15.07,0.14,55.96,Medium,[]
1,39.0,10.0,8.62,5.77,5.0,15.0,11.0,5.0,13.25,0.54,82.22,High,[]
2,34.0,13.0,NaN,4.03,5.0,2.0,18.0,9.0,11.18,1.54,61.77,Medium,[daily_work_hours]
3,30.0,1.0,6.85,6.47,2.0,15.0,26.0,1.0,11.14,0.96,54.98,Medium,[]
4,27.0,7.0,4.24,5.80,NaN,9.0,17.0,7.0,8.05,0.36,27.90,Low,[caffeine_intake]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,25.0,14.0,6.53,7.43,5.0,1.0,7.0,5.0,7.55,1.67,20.10,Low,[]
6996,30.0,19.0,10.15,5.60,3.0,4.0,9.0,0.0,12.11,0.96,41.71,Medium,[]
6997,44.0,8.0,8.56,6.80,5.0,18.0,3.0,4.0,11.62,1.04,80.15,High,[]
6998,25.0,2.0,4.39,5.40,6.0,6.0,4.0,5.0,8.24,0.82,41.44,Medium,[]


In [10]:
count = (data["nan_cols"].str.len() == 0).sum()
count

np.int64(5490)

In [11]:
data = data.dropna(subset=["burnout_level"])
num_cols = data.select_dtypes(include='number')
data[num_cols.columns] = num_cols.fillna(num_cols.mean())

In [12]:
data = data.drop(columns=["nan_cols"])

In [13]:
data

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
0,26.0,12.0,10.330000,4.45,2.000000,11.0,4.0,1.0,15.07,0.14,55.96,Medium
1,39.0,10.0,8.620000,5.77,5.000000,15.0,11.0,5.0,13.25,0.54,82.22,High
2,34.0,13.0,8.989003,4.03,5.000000,2.0,18.0,9.0,11.18,1.54,61.77,Medium
3,30.0,1.0,6.850000,6.47,2.000000,15.0,26.0,1.0,11.14,0.96,54.98,Medium
4,27.0,7.0,4.240000,5.80,3.546482,9.0,17.0,7.0,8.05,0.36,27.90,Low
...,...,...,...,...,...,...,...,...,...,...,...,...
6995,25.0,14.0,6.530000,7.43,5.000000,1.0,7.0,5.0,7.55,1.67,20.10,Low
6996,30.0,19.0,10.150000,5.60,3.000000,4.0,9.0,0.0,12.11,0.96,41.71,Medium
6997,44.0,8.0,8.560000,6.80,5.000000,18.0,3.0,4.0,11.62,1.04,80.15,High
6998,25.0,2.0,4.390000,5.40,6.000000,6.0,4.0,5.0,8.24,0.82,41.44,Medium


In [14]:
len(data)

6860

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["burnout_level"]  # for classification
)

In [17]:
train_df

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
1213,24.0,14.0,4.130000,5.28,0.0,8.0,17.0,7.000000,8.450000,0.80,28.62,Low
2810,25.0,9.0,13.420000,4.87,5.0,10.0,15.0,9.000000,16.050000,0.87,100.00,High
5704,30.0,7.0,8.989003,8.67,0.0,8.0,3.0,1.000000,15.470000,1.40,35.35,Medium
5263,33.0,11.0,11.910000,5.44,2.0,8.0,20.0,5.000000,16.730000,0.63,71.17,High
3189,35.0,12.0,8.989003,5.52,4.0,15.0,28.0,4.000000,16.960000,1.05,85.01,High
...,...,...,...,...,...,...,...,...,...,...,...,...
5355,37.0,17.0,11.890000,4.08,0.0,17.0,14.0,4.548334,11.963314,0.45,93.06,High
3158,28.0,8.0,5.870000,5.65,2.0,8.0,16.0,2.000000,7.950000,0.26,13.98,Low
1893,33.0,12.0,10.800000,7.88,6.0,16.0,21.0,2.000000,13.300000,0.22,75.14,High
3789,32.0,12.0,6.730000,8.09,3.0,7.0,19.0,4.000000,9.640000,0.41,27.46,Low


In [18]:
test_df

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
2979,29.000000,1.0,11.58,5.45,3.0,6.0,17.000000,1.000000,13.04,1.34,46.99,Medium
4530,27.000000,8.0,6.45,4.94,2.0,9.0,8.000000,6.000000,9.09,1.93,36.38,Medium
2639,23.000000,5.0,7.22,8.24,2.0,6.0,14.425915,3.000000,10.72,0.57,30.24,Low
4529,29.000000,6.0,7.14,7.12,5.0,3.0,2.000000,0.000000,11.40,0.19,22.88,Low
4386,41.000000,19.0,7.19,5.16,6.0,1.0,14.000000,8.000000,9.57,0.97,29.55,Low
...,...,...,...,...,...,...,...,...,...,...,...,...
5631,32.000000,17.0,7.49,8.99,3.0,1.0,3.000000,9.000000,12.08,1.10,42.47,Medium
1063,32.132272,4.0,6.11,6.16,2.0,13.0,1.000000,4.548334,8.87,0.60,48.67,Medium
3403,25.000000,12.0,5.94,6.36,6.0,19.0,24.000000,9.000000,7.30,0.04,82.84,High
5959,25.000000,3.0,6.57,8.91,0.0,6.0,1.000000,9.000000,9.14,0.61,44.34,Medium


In [25]:
from src.constants import *
from src.utils.common import read_yaml,create_directories

create_directories([data_transformation_config.root_dir])

[2026-04-16 02:45:51,965: INFO: common: created directory at: artifacts/data_transformation]


In [26]:
train_df.to_csv(f"{data_transformation_config.root_dir}/train.csv", index=False)
test_df.to_csv(f"{data_transformation_config.root_dir}/test.csv", index=False)